제 1유형

1. 각 제품보고서별 처리 시간(처리시각과 신고시각의 차이) 칼럼(초단위)을 생성 후 공장 별 처리 시간의 평균을 산출하시오. 산출된 결과를 바탕으로 평균 처리 시간이 3번째로 적은 공장명을 구하시오.

2. STATION_ADDR1 변수에서 구 정보만 추출한 후, 마포구, 성동구의 평균 이동 거리를 구하시오.

3. 분기별 총 판매량(제품A~E 합계)의 월평균을 구하고, 월평균이 최대인 연도와 분기를 구하시오.

In [10]:
import pandas as pd
df = pd.read_csv('7_1_1.csv')
#print(df.head())

def make_datetime(date_col, time_col):
    date_str = date_col.astype(str)
    time_str = time_col.astype(str).str.zfill(6)
    return pd.to_datetime(date_str + time_str, format='%Y%m%d%H%M%S')

df['신고일시'] = make_datetime(df['신고일자'], df['신고시각'])
df['처리일시'] = make_datetime(df['처리일자'], df['처리시각'])

df['처리시간'] = (df['처리일시'] - df['신고일시']).dt.total_seconds()
#print(df['처리시간'].head())

factory_avg = df.groupby('공장명')['처리시간'].mean().sort_values(ascending=False)

result = factory_avg.index[2]
print(result)


공장D


In [14]:
# 2. STATION_ADDR1 변수에서 구 정보만 추출한 후, 마포구, 성동구의 평균 이동 거리를 구하시오.
import pandas as pd
df = pd.read_csv('7_1_2.csv')
#print(df.head())

# r'(\S+구)' 의미: 공백이 아닌(\S) 글자가 1개 이상(+) 있고 '구'로 끝나는 것
df['구'] = df['STATION_ADDR1'].str.extract(r'(\S+구)')
cols = ['마포구', '성동구']
target = df[df['구'].isin(cols)]['dist'].mean()
print(df['구'].head())
print(target)



0    구로구
1     중구
2    마포구
3    성동구
4    성동구
Name: 구, dtype: object
3113.6666666666665


In [17]:
#3. 분기별 총 판매량(제품A~E 합계)의 월평균을 구하고, 월평균이 최대인 연도와 분기를 구하시오.
import pandas as pd
df = pd.read_csv('7_1_3.csv')
#print(df.head())

df['total'] = df['제품A'] + df['제품B'] + df['제품C'] + df['제품D'] + df['제품E']

q1 = df.groupby('기간')['total'].mean().sort_values(ascending=False)
print(q1)

기간
2018년_9월     3957.0
2019년_2월     3490.0
2019년_9월     3417.0
2019년_7월     3333.0
2018년_12월    3293.0
2019년_1월     3226.0
2018년_1월     3184.0
2018년_3월     3045.0
2018년_10월    3038.0
2018년_2월     2963.0
2019년_11월    2880.0
2018년_7월     2870.0
2018년_8월     2853.0
2018년_5월     2641.0
2019년_12월    2637.0
2019년_8월     2605.0
2018년_11월    2546.0
2019년_4월     2412.0
2018년_4월     2398.0
2019년_3월     2026.0
2019년_10월    1851.0
2018년_6월     1771.0
2019년_5월     1683.0
2019년_6월     1342.0
Name: total, dtype: float64


제 2유형

훈련 데이터로 학습한 모델을 테스트 데이터에 적용하여 예측한 결과를 제출하시오.(분류 예측값 제출)

% 제출 형식은 ID, pred 두 칼럼만 존재해야 한다.(평가 지표: macro f1-score)

In [19]:
import pandas as pd
train = pd.read_csv('7_2_train.csv')
test = pd.read_csv('7_2_test.csv')

#print(test.info())
# 제거: ID, Target

X = train.drop(['ID', 'Target'], axis=1)
y = train['Target']
X_submit = test.drop(['ID', 'Target'], axis=1)

cols = X.select_dtypes(include='object').columns
#print(cols)
ct = pd.concat([X, X_submit])
encoded = pd.get_dummies(ct, columns=cols)
X = encoded.iloc[:len(X)]
X_submit = encoded.iloc[len(X):]
print(X.info())

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_val)

from sklearn.metrics import f1_score
score = f1_score(y_val, pred, average='macro')
print(round(score, 3))

model = RandomForestClassifier(random_state=42)
model.fit(X, y)
pred_final = model.predict(X_submit)

result = pd.DataFrame({
    'ID': test['ID'],
    'pred': pred_final
})

#result.to_csv('result.csv', index=False)
print(result.head())

<class 'pandas.core.frame.DataFrame'>
Index: 309 entries, 0 to 308
Columns: 143 entries, level_0 to Gender_1_
dtypes: bool(138), int64(5)
memory usage: 56.1 KB
None
0.487
     ID      pred
0  1903  Graduate
1   796  Graduate
2  1060  Graduate
3   791   Dropout
4  3416  Graduate


제 3유형

<3-1 학습 프로그램 효과 조사, 학습 전 / 학습 후 시험 점수 측정>

1.1. 학습 전과 학습 후의 시험 점수의 평균과 표준편차를 구하시오.(소수점 둘째 자리까지 반올림)

1.2. 학습 전후의 점수 차이가 유의미한 지 검정하기 위해 대응표본 t-검정을 수행하고, 검정 통계량을 계산하시오.(소수점 둘째 자리까지 반올림)

1.3. p-value를 바탕으로 유의수준 5%에서 귀무가설의 기각/채택 여부를 결정하시오.(p-value는 소수점 셋째 자리까지 반올림)

<3-2 회사 조사 고객 데이터는 100개의 샘플로 구성, 각 샘플에는 고객의 나이, 소득, 가족 수, 제품 구매 여부 포함. 로지스틱 회귀 분석 통해 고객의 제품 구매 여부를 예측(임곗값 0.5 기준)>

2.1. 로지스틱 회귀 분석을 수행하고, 소득 변수의 오즈비를 계산하시오.

2.2. train 데이터 기준의 잔차 이탈도(Residual Deviance)를 계산하시오.

2.3. test 데이터로 오분류율을 계산하시오.


In [ ]:
#<3-1 학습 프로그램 효과 조사, 학습 전 / 학습 후 시험 점수 측정>
#1.1. 학습 전과 학습 후의 시험 점수의 평균과 표준편차를 구하시오.(소수점 둘째 자리까지 반올림)
#1.2. 학습 전후의 점수 차이가 유의미한 지 검정하기 위해 대응표본 t-검정을 수행하고, 검정 통계량을 계산하시오.(소수점 둘째 자리까지 반올림)
#1.3. p-value를 바탕으로 유의수준 5%에서 귀무가설의 기각/채택 여부를 결정하시오.(p-value는 소수점 셋째 자리까지 반올림)
import pandas as pd
df = pd.read_csv('7_3_1.csv')
print(df.head())

before_mean = df['before'].mean()
after_mean = df['after'].mean()
before_std = df['before'].std()
after_std = df['after'].std()

print(round(before_mean, 2))
print(round(after_mean, 2))
print(round(before_std, 2))
print(round(after_std, 2))

from scipy.stats import ttest_rel
stat, p_val = ttest_rel(df['before'], df['after'])
print(round(stat, 2))
print(round(p_val, 3))
# 기각

      before      after
0  87.640523  88.163191
1  74.001572  80.936085
2  79.787380  82.233354
3  92.408932  91.505771
4  88.675580  93.534669
71.41
76.3
11.37
11.94
-7.9
0.0


In [2]:
#<3-2 회사 조사 고객 데이터는 100개의 샘플로 구성, 각 샘플에는 고객의 나이, 소득, 가족 수, 제품 구매 여부 포함. 로지스틱 회귀 분석 통해 고객의 제품 구매 여부를 예측(임곗값 0.5 기준)>

#2.1. 로지스틱 회귀 분석을 수행하고, 소득 변수의 오즈비를 계산하시오.
#2.2. train 데이터 기준의 잔차 이탈도(Residual Deviance)를 계산하시오.
#2.3. test 데이터로 오분류율을 계산하시오.
import pandas as pd
train = pd.read_csv('7_3_2_train.csv')
test = pd.read_csv('7_3_2_test.csv')
#print(train.head())

from statsmodels.formula.api import logit
model = logit('purchase ~ age + income + family_members', data=train).fit()
#print(model.summary())

import numpy as np
coef = model.params['income']
odds_ratio = np.exp(coef)
print(round(odds_ratio, 4))  

pred = model.predict(test)
pred_out = np.where(pred > 0.5, 1, 0)
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test['purchase'], pred_out)
mis_classification = 1 - accuracy
print(mis_classification)
#print(pred_out)

Optimization terminated successfully.
         Current function value: 0.676174
         Iterations 4
1.0
0.35
